## 📁 Dataset Folder Structure

```
thai_food_dataset/
├── ผัดไทย/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (50-200 images)
├── ข้าวมันไก่/
│   ├── image1.jpg
│   └── ...
├── แกงเขียวหวาน/
├── ข้าวผัด/
├── ส้มตำ/
├── ผัดกะเพรา/
├── ข้าวซอย/
├── ลาบหมู/
├── ต้มยำกุ้ง/
└── ข้าวไข่เจียว/
```

**Tip:** Download images from Google Images, Unsplash, or your own photos. Aim for 50-200 images per class.

In [ ]:
# Install required packages
!pip install tensorflowjs -q

In [ ]:
# type: ignore
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import os
import json

## 📤 Upload Your Dataset

Upload your `thai_food_dataset.zip` file containing all food images organized by folders.

In [ ]:
from google.colab import files
import zipfile
import os

# Upload dataset zip file
print("📤 Please upload your thai_food_dataset.zip file:")
uploaded = files.upload()

# Extract the zip file
for filename in uploaded.keys():
    print(f"\n📦 Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("✅ Extraction complete!")

# Debug: List what was extracted
print("\n📋 Contents after extraction:")
for item in os.listdir('.'):
    item_type = "📁" if os.path.isdir(item) else "📄"
    if not item.startswith('.'):
        print(f"  {item_type} {item}")

# Find the dataset directory - be flexible with naming
dataset_dir = None

# Check for any directory that looks like our dataset
for item in sorted(os.listdir('.')):
    if not item.startswith('.') and item not in ['sample_data']:
        item_path = os.path.join('.', item)
        if os.path.isdir(item_path):
            # Check if this directory contains food class folders
            try:
                contents = os.listdir(item_path)
                subdirs = [d for d in contents if os.path.isdir(os.path.join(item_path, d))]
                if len(subdirs) >= 5:  # Should have many subdirectories (our food classes)
                    dataset_dir = item
                    print(f"\n✅ Found dataset with {len(subdirs)} classes: {item}")
                    break
            except:
                pass

if not dataset_dir:
    print("\n⚠️  Could not find dataset folder automatically")
    print("Available directories:")
    for item in os.listdir('.'):
        if os.path.isdir(item) and not item.startswith('.'):
            print(f"  - {item}")
    dataset_dir = 'thai_food_dataset'  # Default

print(f"\n📁 Using: {dataset_dir}")

# List food classes
if dataset_dir and os.path.isdir(dataset_dir):
    print(f"\n📊 Food classes found:")
    try:
        class_dirs = sorted(os.listdir(dataset_dir))
        for idx, class_name in enumerate(class_dirs, 1):
            class_path = os.path.join(dataset_dir, class_name)
            if os.path.isdir(class_path):
                images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp'))]
                num_images = len(images)
                print(f"  {idx}. {class_name}: {num_images} images")
    except Exception as e:
        print(f"  Error: {e}")


## ⚙️ Configure Training Parameters

In [ ]:
# Training configuration
IMG_SIZE = (224, 224)  # MobileNetV2 default input size
BATCH_SIZE = 32
EPOCHS = 20  # Adjust based on your dataset size
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2

# Class names in the order they appear in your dataset
CLASS_NAMES = [
    'pad_thai',
    'khao_man_gai',
    'green_curry',
    'fried_rice',
    'papaya_salad',
    'basil_stir_fry',
    'khao_soi',
    'larb_moo',
    'tom_yum_goong',
    'omelet_rice'
]

NUM_CLASSES = len(CLASS_NAMES)
print(f"Number of classes: {NUM_CLASSES}")

## 🖼️ Data Preparation & Augmentation

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)

# Only rescaling for validation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

# Load training data
train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# Load validation data
validation_generator = val_datagen.flow_from_directory(
    dataset_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"\n✅ Training samples: {train_generator.samples}")
print(f"✅ Validation samples: {validation_generator.samples}")
print(f"\n📋 Class indices: {train_generator.class_indices}")

## 🏗️ Build Model with MobileNetV2

In [ ]:
# Load pre-trained MobileNetV2 (without top classification layer)
base_model = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model layers initially
base_model.trainable = False

# Build model
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

model.summary()

## 🎯 Train the Model

In [ ]:
# DEBUG: Check what's in the dataset directory
import os

print("🔍 DEBUGGING DATASET...\n")

if not os.path.isdir(dataset_dir):
    print(f"❌ Dataset directory does not exist: {dataset_dir}")
    print(f"Available directories: {os.listdir('.')}")
else:
    print(f"✅ Dataset directory exists: {dataset_dir}")
    
    total_images = 0
    for class_name in sorted(os.listdir(dataset_dir)):
        class_path = os.path.join(dataset_dir, class_name)
        
        if os.path.isdir(class_path):
            all_files = os.listdir(class_path)
            image_files = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp'))]
            print(f"  📂 {class_name}/")
            print(f"     Total files: {len(all_files)}")
            print(f"     Image files: {len(image_files)}")
            if image_files:
                print(f"     Examples: {', '.join(image_files[:3])}")
            elif all_files:
                print(f"     Files: {all_files[:5]}")
            total_images += len(image_files)
        else:
            print(f"  ❌ {class_name} is not a directory!")
    
    print(f"\n📊 Total images found: {total_images}")
    if total_images == 0:
        print("\n⚠️  NO IMAGES FOUND! Check:")
        print("  1. Images are in the correct folders")
        print("  2. File extensions are .jpg, .jpeg, or .png")
        print("  3. Files are actually image files, not something else")

In [ ]:
# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7
    )
]

# Train model
print("\n🚀 Starting training...\n")
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("\n✅ Initial training complete!")

## 🔓 Fine-tune the Model

In [ ]:
# Unfreeze the base model for fine-tuning
base_model.trainable = True

# Freeze early layers, only train last layers
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE/10),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

# Fine-tune
print("\n🔥 Starting fine-tuning...\n")
history_fine = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10,
    callbacks=callbacks
)

print("\n✅ Fine-tuning complete!")

## 📊 Visualize Training Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

# Top-3 Accuracy
axes[2].plot(history.history['top_3_accuracy'], label='Train Top-3')
axes[2].plot(history.history['val_top_3_accuracy'], label='Val Top-3')
axes[2].set_title('Top-3 Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

# Final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_top3_acc = history.history['val_top_3_accuracy'][-1]

print(f"\n📈 Final Training Accuracy: {final_train_acc:.4f}")
print(f"📈 Final Validation Accuracy: {final_val_acc:.4f}")
print(f"📈 Final Top-3 Accuracy: {final_top3_acc:.4f}")

## 💾 Export Model for TensorFlow.js

In [ ]:
import tensorflowjs as tfjs

# Create output directory
output_dir = 'tfjs_model'
os.makedirs(output_dir, exist_ok=True)

# Save model in TensorFlow.js format (LayersModel)
print("\n💾 Converting model to TensorFlow.js format...")
tfjs.converters.save_keras_model(model, output_dir)
print(f"✅ Model saved to {output_dir}/")

# Save class names
class_names_path = os.path.join(output_dir, 'class_names.json')
with open(class_names_path, 'w', encoding='utf-8') as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)
print(f"✅ Class names saved to {class_names_path}")

# Save training metadata
metadata = {
    'model_type': 'MobileNetV2',
    'num_classes': NUM_CLASSES,
    'input_size': IMG_SIZE,
    'training_accuracy': float(final_train_acc),
    'validation_accuracy': float(final_val_acc),
    'top_3_accuracy': float(final_top3_acc),
    'classes': CLASS_NAMES
}

metadata_path = os.path.join(output_dir, 'metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✅ Metadata saved to {metadata_path}")

print("\n📦 Files to download:")
for file in sorted(os.listdir(output_dir)):
    file_path = os.path.join(output_dir, file)
    if os.path.isfile(file_path):
        size = os.path.getsize(file_path) / 1024  # KB
        print(f"  - {file} ({size:.2f} KB)")

## 📥 Download Model Files

In [ ]:
# Zip the model folder for easy download
import shutil

zip_filename = 'thai_food_model_tfjs'
print(f"\n📦 Creating zip file: {zip_filename}.zip")
shutil.make_archive(zip_filename, 'zip', output_dir)
print("✅ Zip file created!")

# Download the zip file
print("\n⬇️ Downloading model files...")
from google.colab import files
files.download(f"{zip_filename}.zip")
print("\n✅ Download complete!")

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🎉 MODEL TRAINING COMPLETE!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 Next Steps:

1. Extract the downloaded thai_food_model_tfjs.zip
2. Copy all files to: MFU Longevity Passport/public/model/
3. Replace these files:
   - model.json
   - group1-shard*.bin (all shard files)
   - class_names.json

4. Update src/config/foodClasses.ts with new class names

5. Test the model in your app!

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

## 🧪 Test the Model (Optional)

In [ ]:
# Test with a few validation images
from tensorflow.keras.preprocessing import image
import random

def predict_image(img_path, model, class_names, img_size=IMG_SIZE, top_k=3):
    """Predict food class for a single image"""
    # Load and preprocess image
    img = image.load_img(img_path, target_size=img_size)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    # Predict
    predictions = model.predict(img_array, verbose=0)
    
    # Get top K predictions
    top_indices = predictions[0].argsort()[-top_k:][::-1]
    
    # Display image and predictions
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Input Image')
    
    y_pos = np.arange(top_k)
    probs = predictions[0][top_indices]
    labels = [class_names[i] for i in top_indices]
    
    axes[1].barh(y_pos, probs)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(labels)
    axes[1].set_xlabel('Confidence')
    axes[1].set_title(f'Top {top_k} Predictions')
    axes[1].set_xlim(0, 1)
    
    for i, (label, prob) in enumerate(zip(labels, probs)):
        axes[1].text(prob, i, f' {prob:.2%}', va='center')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n🎯 Predicted: {labels[0]} ({probs[0]:.2%} confidence)")
    return labels[0], probs[0]

# Test with random validation images
print("\n🧪 Testing model with validation images...\n")

# Get random images from validation set
test_images = []
for class_name in CLASS_NAMES:
    class_path = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_path):
        images = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if images:
            test_images.append(os.path.join(class_path, random.choice(images)))

# Test 3 random images
for img_path in random.sample(test_images, min(3, len(test_images))):
    true_class = os.path.basename(os.path.dirname(img_path))
    print(f"\nTrue class: {true_class}")
    predict_image(img_path, model, CLASS_NAMES)
    print("─" * 50)